In [365]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re
import ollama
import numpy as np

In [366]:
loader = PyPDFLoader('data/2312.10997v5.pdf')
loader = PyPDFLoader('data/10201979.pdf')
loader = PyPDFLoader('data/2509.23988v3.pdf')
loader = PyPDFLoader('data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf')
pages = loader.load()


In [367]:
pages[0].metadata

{'producer': 'pdfTeX-1.40.24',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2025-04-15T21:43:08+05:30',
 'moddate': '2025-04-17T21:53:13-04:00',
 'trapped': '/False',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4',
 'keywords': '',
 'title': '',
 'subject': '',
 'author': '',
 'source': 'data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf',
 'total_pages': 28,
 'page': 0,
 'page_label': '64087'}

In [368]:
pages[0].page_content

'Received 13 February 2025, accepted 28 March 2025, date of publication 1 April 2025, date of current version 18 April 2025.\nDigital Object Identifier 10.1 109/ACCESS.2025.3556973\nExploring the Potential of Offline LLMs in Data\nScience: A Study on Code Generation\nfor Data Analysis\nANASTASIOS NIKOLAKOPOULOS\n 1, ANTONIOS LITKE\n 1, ALEXANDROS PSYCHAS\n 1,\nELENI VERONI\n 2, AND THEODORA VARVARIGOU 1\n1School of Electrical and Computer Engineering, Institute of Communication and Computer Systems, National Technical University of Athens, 157 73 Zografou,\nGreece\n2Research and Innovation Development Department, Netcompany-Intrasoft S.A., 15125 Luxembourg, Luxembourg\nCorresponding author: Anastasios Nikolakopoulos (tasosnikolakop@mail.ntua.gr)\nThis work was supported in part by the European Commission through the Horizon Europe Program through the Project ‘‘Reliable\nbiomeTric tEchNologies to asSist Police authorities in cOmbating terrorism and oRganized crime (TENSOR)’’ under Grant

In [369]:
page = pages[0]
page = page.model_dump_json()

In [370]:
page

'{"id":null,"metadata":{"producer":"pdfTeX-1.40.24","creator":"LaTeX with hyperref","creationdate":"2025-04-15T21:43:08+05:30","moddate":"2025-04-17T21:53:13-04:00","trapped":"/False","ptex.fullbanner":"This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4","keywords":"","title":"","subject":"","author":"","source":"data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf","total_pages":28,"page":0,"page_label":"64087"},"page_content":"Received 13 February 2025, accepted 28 March 2025, date of publication 1 April 2025, date of current version 18 April 2025.\\nDigital Object Identifier 10.1 109/ACCESS.2025.3556973\\nExploring the Potential of Offline LLMs in Data\\nScience: A Study on Code Generation\\nfor Data Analysis\\nANASTASIOS NIKOLAKOPOULOS\\n 1, ANTONIOS LITKE\\n 1, ALEXANDROS PSYCHAS\\n 1,\\nELENI VERONI\\n 2, AND THEODORA VARVARIGOU 1\\n1School of Electrical and Computer Engineering, Institut

In [371]:
#page = {
 #   "id": None,
    #"metadata": {
      #  "producer": "Microsoft® Word 2016",
      #  "creator": "Microsoft® Word 2016",
      #  "creationdate": "2026-05-05T22:07:15+05:30",
       # "author": "TechnoScience Academy;http://technoscienceacademy.com",
       # "keywords": "IJSRCSEIT; http:/ijsrcseit.com",
       # "moddate": "2026-05-05T22:07:15+05:30",
       # "subject": "Research Paper",
      #  "title": "International Journal of Scientific Research in Computer Science, Engineering and Information Technology",
      #  "source": "data/Mr. Himanshu Barhaiya - Advances in Retrieval-Augmented Generation for Scientific and Technical QA A Review [2026].pdf",
      #  "total_pages": 12,
      #  "page": 0,
       # "page_label": "1"
   # },
   # "page_content": "Copyright © 202 6 The Author(s)  : This is an open access article under the CC BY license \\n(http://creativecommons.org/licenses/by/4.0/) \\n \\n \\n \\nInternational Journal of Scientific Research in Computer Science, Engineering \\nand Information Technology \\nPeer Reviewed and Refereed International Scientific Research Journal \\nISSN : 2456-3307 \\nAvailable Online at : ijsrcseit.com \\n \\ndoi : https://doi.org/10.32628/IJSRCSEIT \\n \\n \\n \\n \\n  \\n \\n \\n \\n31 \\nAdvances in Retrieval-Augmented Generation for Scientific and \\nTechnical QA: A Review    \\nMr. Himanshu Barhaiya \\nDepartment of Computer Science and Engineering, Lakshmi Narain College of Technology, Bhopal, Madhya \\nPradesh, India \\nA R T I C L E I N F O  A B S T R A C T \\nArticle History: \\nAccepted : 01 April 2026 \\nPublished: 03 May 2026 \\n \\n Retrieval-Augmented Generation (RAG) works by using big language models \\ntogether with external knowledge retrieval systems to increase quality and \\ncontext of text it generates. Traditional LLMs are limited because the information \\nthey provide is drawn from static data, making it hard for them to keep up. RAG \\ndeals with this problem by finding appropriate documents or segments from an \\nexternal knowledge source and sending them to the generative model for \\nprocessing. This approach to fetching information allo ws RAG to respond with \\nmore context, better facts and greater understanding. Usually, the process \\nincludes a retriever that selects the best text groups from the database related to \\nthe query, followed by a generator that combines the input and the selected \\ncontent to produce coherent text. RAG has performed well in processes like \\nopen-domain question answering, summarization and analyzing documents in an \\nenterprise environment, mainly in domains such as healthcare, law and finance. \\nBecause it does not need  retraining to process fresh data, it’s very efficient and \\nsuitable for real-world situations. Building RAG into NLP is important as it helps \\nmake language models more understandable, explanatory and knowledgeable. \\nKeywords—Retrieval-Augmented Generation, Large Language Models, Natural \\nLanguage Processing, Information Retrieval, Question Answering, Document \\nUnderstanding, Contextual Generation \\n \\nPublication Issue \\nVolume 12, Issue 3 \\nMay-June-2026 \\n \\nPage Number  \\n31-42 \\n \\nIntroduction \\nRetrieval-Augmented Generation (RAG) is a major \\napproach in NLP that helps unite information \\nretrieval and the process of text production. RAG \\nbenefits from combining knowledge -finding \\nretrievers and large language models (LLMs) for the \\ngeneration of responses , which helps with factual \\ncorrectness, and avoids hallucination [1][2] and makes \\nthe answers more tailored to the specific user [3]. This \\nkind of architecture works very well in domains \\nwhere the use of up-to-date, contextual information is \\nimportant. Evaluating RAG systems ensures that both \\nretrieval methods and generative models are \\nintegrated appropriately [4]. The standard evaluation",
   # "type": "Document"
#}


In [372]:
#page['page_content']

In [373]:
#pages

In [374]:
#print(pages[4].page_content)

### Text Cleaning

In [375]:
def cleanText(text): 
    cleaned_text = re.sub(r'\s+', ' ', text) # clears the spacess
    cleaned_text = cleaned_text.replace('-', " ")
    cleaned_text = cleaned_text.replace(' -', " ")
    cleaned_text = cleaned_text.replace('•', ";")
    cleaned_text = cleaned_text.replace("   ", " ")
    cleaned_text = cleaned_text.replace("__", "")
    return cleaned_text

In [376]:
for page in pages:
    page.page_content = cleanText(page.page_content)

In [377]:
pages

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-15T21:43:08+05:30', 'moddate': '2025-04-17T21:53:13-04:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'keywords': '', 'title': '', 'subject': '', 'author': '', 'source': 'data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf', 'total_pages': 28, 'page': 0, 'page_label': '64087'}, page_content='Received 13 February 2025, accepted 28 March 2025, date of publication 1 April 2025, date of current version 18 April 2025. Digital Object Identifier 10.1 109/ACCESS.2025.3556973 Exploring the Potential of Offline LLMs in Data Science: A Study on Code Generation for Data Analysis ANASTASIOS NIKOLAKOPOULOS 1, ANTONIOS LITKE 1, ALEXANDROS PSYCHAS 1, ELENI VERONI 2, AND THEODORA VARVARIGOU 1 1School of Electrical and Computer Engineering, Institute of

In [378]:
pages[: 10]

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-15T21:43:08+05:30', 'moddate': '2025-04-17T21:53:13-04:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'keywords': '', 'title': '', 'subject': '', 'author': '', 'source': 'data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf', 'total_pages': 28, 'page': 0, 'page_label': '64087'}, page_content='Received 13 February 2025, accepted 28 March 2025, date of publication 1 April 2025, date of current version 18 April 2025. Digital Object Identifier 10.1 109/ACCESS.2025.3556973 Exploring the Potential of Offline LLMs in Data Science: A Study on Code Generation for Data Analysis ANASTASIOS NIKOLAKOPOULOS 1, ANTONIOS LITKE 1, ALEXANDROS PSYCHAS 1, ELENI VERONI 2, AND THEODORA VARVARIGOU 1 1School of Electrical and Computer Engineering, Institute of

In [379]:
len(pages[0].page_content)

4256

### Chunking
 - splitting text information into smaller sizes

In [380]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap = 100)

In [381]:
chunks = splitter.split_documents(pages)

In [382]:
chunks[0:10]

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-15T21:43:08+05:30', 'moddate': '2025-04-17T21:53:13-04:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'keywords': '', 'title': '', 'subject': '', 'author': '', 'source': 'data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf', 'total_pages': 28, 'page': 0, 'page_label': '64087'}, page_content='Received 13 February 2025, accepted 28 March 2025, date of publication 1 April 2025, date of current version 18 April 2025. Digital Object Identifier 10.1 109/ACCESS.2025.3556973 Exploring the'),
 Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-15T21:43:08+05:30', 'moddate': '2025-04-17T21:53:13-04:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.2

In [383]:
len(chunks)

1394

### Embeddings

In [384]:
EMBEDDING_MODEL = 'nomic-embed-text-v2-moe'
def createEmbedding(text):
    response = ollama.embed(
    model = EMBEDDING_MODEL,
    input = text ,
    )
    return response.embeddings

In [385]:
doc_embeddings = []

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding)

KeyboardInterrupt: 

In [ ]:
len(doc_embeddings)

425

In [ ]:
user_query = input('Ask anything about the Document...')


Tell me  about LLMs and RAG?


In [ ]:
print(user_query)   

In [ ]:
user_query_embedding  = createEmbedding(user_query)

In [ ]:
print(user_query_embedding)

[[0.0068232142, -0.03543058, -0.014502778, -0.021702569, 0.009820116, -0.05984446, 0.0143169, -0.025361996, -0.047412157, 0.090123184, -0.093021646, -0.014957851, 0.019336965, 0.010958245, -0.015593868, -0.048828352, 0.01664014, 0.060569435, -0.000411548, 0.031396683, -0.00522633, 0.01364962, -0.050901737, 0.021131009, 0.016167352, 0.022282928, -0.036700733, -0.025896104, 0.0765768, 0.028320467, -0.0071670054, 0.011302114, 0.056766037, 0.022949446, -0.013504247, -0.011756488, -0.055002805, -0.012562897, -0.04354451, -0.016530808, -0.009620711, 0.013937716, 0.040618215, -0.049469832, -0.026881017, -0.009103739, -0.11279582, 0.0064782286, -0.011440036, -0.0038816356, 0.004529836, 0.0034179664, 0.05288688, -0.059294198, 0.024478992, -0.053785548, -0.07905774, -0.071948946, 0.01025211, -0.0014902158, -0.061053645, -0.07274471, 0.024577707, 0.048375078, -0.051402032, 0.059746675, 0.078358024, 0.010788003, -0.025730593, 0.06132808, -0.051998448, -0.008971593, -0.012010009, -0.031429894, -0.0

In [ ]:
above_zero_point_six = []

for index, doc_embedding in enumerate(doc_embeddings):
    score = cosine_similarity(doc_embedding, user_query_embedding)
    if score >= 0.6:
        result = (index, score)
        above_zero_point_six.append(result)

In [ ]:
above_zero_point_six

[]

In [ ]:
largest = max(above_zero_point_six, key=lambda x: x[1])
largest

ValueError: max() iterable argument is empty

In [ ]:
response_to_client = chunks[largest[0]]
print(response_to_client.page_content)

NameError: name 'largest' is not defined

In [ ]:
chunk[above_zero_point_six[2][0]].page_content

### Evaluations

In [ ]:
a = np.array([1,2,3])
b = np.array([1,2,3])

In [ ]:
dot_sim = np.dot(a,b)

In [ ]:
cos_sim = np.dot(a,b) / (np.linalg.norm(a)* np.linalg.norm(b))

In [ ]:
euclidean_dist = np.linalg.norm(a - b)

In [ ]:
output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
output_cos = f" COSINE SIM_SCORE: {cos_sim} "
output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

print(f""" 
{output_dot} \n 
{output_cos} \n 
{output_eucl} \n 
""")

In [ ]:
def similarityChecks(text_one_embeding, text_two_embeding):
    # Dot Product Similarity
    dot_sim = np.dot(text_one_embeding, text_two_embeding)
    # Cosine Similarity
    cos_sim = np.dot(text_one_embeding, text_two_embeding) / (np.linalg.norm(text_one_embeding) * np.linalg.norm(text_two_embeding))
    # Euclidean Distance (Lower is better)
    euclidean_dist = np.linalg.norm(np.array(text_one_embeding) - np.array(text_two_embeding))   

    output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
    output_cos = f" COSINE SIM_SCORE: {cos_sim} "
    output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

    print(f" {output_dot} \n {output_cos} \n {output_eucl} \n ")

In [ ]:
user_query = input("ask me anything...")